In [1]:
import chromedriver_autoinstaller
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options as ChromeOptions
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC

import requests
import pandas as pd
from datetime import date, timedelta
from functools import reduce
import os
from shutil import which
from bs4 import BeautifulSoup

chromedriver_autoinstaller.install()

'c:\\Python310\\lib\\site-packages\\chromedriver_autoinstaller\\157\\chromedriver.exe'

In [4]:
def _setup_chrome_options():
    """Set up Chrome options for local Selenium execution."""
    options = ChromeOptions()
    
    # Keep Chrome visible while running locally.
    # Uncomment this when you want headless execution again.
    # options.add_argument("--headless=new")
    
    options.add_argument("--start-maximized")
    
    return options

In [5]:
url = "https://www.rotowire.com/hockey/nhl-lineups.php"
# url = "https://www.rotowire.com/hockey/nhl-lineups.php?date=tomorrow"

options = _setup_chrome_options()
driver = webdriver.Chrome(options=options)

driver.set_page_load_timeout(20)
driver.set_script_timeout(20)

driver.get(url)

print("Page title:", driver.title)
print("Current URL:", driver.current_url)

Page title: NHL Starting Lineups: Hockey Daily Lineups | RotoWire
Current URL: https://www.rotowire.com/hockey/nhl-lineups.php


In [6]:
WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.CLASS_NAME, "lineups"))
)

print("Lineup container found!")

Lineup container found!


In [7]:
matchups = []

lineups = (
    driver.find_element(By.CLASS_NAME, "lineups")
    .find_elements(By.CLASS_NAME, "is-nhl")
)

print("Number of lineups found:", len(lineups))

Number of lineups found: 7


In [8]:
for lineup in lineups[:-1]:
    goalies = lineup.find_elements(
        By.CLASS_NAME,
        "lineup__player-highlight-name"
    )

    away_goalie = goalies[0].text.strip().split()[-1]
    home_goalie = goalies[1].text.strip().split()[-1]

    teams = lineup.find_element(
        By.CLASS_NAME,
        "lineup__matchup"
    ).find_elements(By.TAG_NAME, "a")

    if len(teams) >= 2:
        away_team = teams[0].text.split(" (")[0]
        home_team = teams[1].text.split(" (")[0]

        matchups.append(
            (away_team, home_team, away_goalie, home_goalie)
        )

matchups

[('Lightning', 'Sabres', 'Vasilevskiy', 'Luukkonen'),
 ('Rangers', 'Flames', 'Korpisalo', 'Cooley'),
 ('Flyers', 'Oilers', 'Vladar', 'Jarry'),
 ('Mammoth', 'Golden Knights', 'Cossa', 'Hart'),
 ('Predators', 'Ducks', 'Annunen', 'Brossoit'),
 ('Penguins', 'Sharks', 'Murashov', 'Askarov')]

In [9]:
driver.quit()

In [10]:
url = "https://moneypuck.com/goalies.htm"

options = _setup_chrome_options()
driver = webdriver.Chrome(options=options)

driver.set_page_load_timeout(20)
driver.set_script_timeout(20)

driver.get(url)

print("Page title:", driver.title)
print("Current URL:", driver.current_url)

Page title: NHL Goalie Advanced Stats 2026-2027
Current URL: https://moneypuck.com/goalies.htm


In [11]:
wait = WebDriverWait(driver, 10)

dropdown = wait.until(
    EC.presence_of_element_located((By.ID, "num_games_type"))
)

select = Select(dropdown)

select.select_by_value("_20")

print("Selected Last 20 Games")

Selected Last 20 Games


In [12]:
wait = WebDriverWait(driver, 10)

dropdown = wait.until(
    EC.presence_of_element_located((By.ID, "num_games_type"))
)

select = Select(dropdown)

select.select_by_value("_20")

print("Selected Last 20 Games")

Selected Last 20 Games


In [13]:
wait.until(
    EC.presence_of_element_located((By.ID, "goaliesTable"))
)

print("Goalie table found!")

Goalie table found!


In [14]:
table_html = driver.find_element(
    By.ID,
    "goaliesTable"
).get_attribute("outerHTML")

print("HTML length:", len(table_html))

HTML length: 180324


In [15]:
driver.quit()

In [16]:
soup = BeautifulSoup(table_html, "html.parser")

rows = soup.find_all("tr")[1:]

data = []

for tr in rows:
    tds = tr.find_all("td")

    if len(tds) < 7:
        continue

    try:
        rank = int(tds[0].get_text(strip=True))
        goalie = tds[2].get_text(strip=True).split()[-1]
        gsa = float(tds[6].get_text(strip=True))
    except:
        continue

    data.append({
        "Rank": rank,
        "Goalie": goalie,
        "GSAx": gsa
    })

goalie_df = pd.DataFrame(data)

goalie_df.head(20)

,Rank,Goalie,GSAx
0,1,Wallstedt,15.9
1,2,Fowler,12.3
2,3,Swayman,10.2
3,4,Saros,9.3
4,5,Kochetkov,8.8
5,6,Ullmark,8.3
6,7,Oettinger,7.6
7,8,Sorokin,7.3
8,9,Tarasov,6.5
9,10,Hofer,6.5


In [17]:
goalie_ranks = dict(
    zip(goalie_df["Goalie"], goalie_df["Rank"])
)

goalie_ranks

{'Wallstedt': 1,
 'Fowler': 2,
 'Swayman': 3,
 'Saros': 4,
 'Kochetkov': 5,
 'Ullmark': 6,
 'Oettinger': 7,
 'Sorokin': 8,
 'Tarasov': 9,
 'Hofer': 10,
 'Talbot': 11,
 'Nedeljkovic': 12,
 'Shesterkin': 13,
 'DiPietro': 14,
 'Varlamov': 15,
 'Woll': 16,
 'Askarov': 17,
 'Hill': 18,
 'Thompson': 19,
 'Cooley': 20,
 'Stolarz': 21,
 'Ellis': 22,
 'Hart': 23,
 'Garand': 24,
 'Markstrom': 25,
 'Levi': 26,
 'Brossoit': 27,
 'Hellebuyck': 28,
 'Murray': 29,
 'Blackwood': 30,
 'DeSmith': 31,
 'Tomkins': 32,
 'Korpisalo': 33,
 'Schmid': 34,
 'Forsberg': 35,
 'Skinner': 36,
 'Postava': 37,
 'Vasilevskiy': 38,
 'Lindgren': 39,
 'Sergeev': 40,
 'Grubauer': 41,
 'Vladar': 42,
 'Annunen': 43,
 'Black': 44,
 'Luukkonen': 45,
 'Martin': 46,
 'Hildeby': 47,
 'Jarry': 48,
 'Dostal': 49,
 'Murashov': 50,
 'Greaves': 51,
 'Daws': 52,
 'Reimer': 53,
 'Husso': 54,
 'Pickard': 55,
 'Knight': 56,
 'Merilinen': 57,
 'Bobrovsky': 58,
 'Silovs': 59,
 'Daccord': 60,
 'Ersson': 61,
 'Stevenson': 62,
 'Cossa': 63,
 

In [18]:
import os
from dotenv import load_dotenv

load_dotenv()

key = os.getenv("NST_API_KEY")

if not key:
    raise ValueError("NST_API_KEY not found. Check your .env file.")

In [30]:
from io import StringIO

# Date range
today = date.today()
start = today - timedelta(days=30)

# Your NST access key
key = "451fe70942969369f896204690d6cd68"

# Build the URL using NST's data subdomain
url = (
    "https://data.naturalstattrick.com/teamtable.php"
    f"?key={key}"
    f"&fromseason=20252026"
    f"&thruseason=20252026"
    f"&stype=2"
    f"&sit=5v5"
    f"&score=all"
    f"&rate=y"
    f"&team=all"
    f"&loc=B"
    f"&gpf=410"
    f"&fd={start}"
    f"&td={today}"
)

print("Requesting NST team table...")

response = requests.get(
    url,
    headers={"nst-key": key},
    timeout=30
)
response.raise_for_status()

html = response.text

print("Status code:", response.status_code)
print("HTML length:", len(html))
print("Contains table:", "<table" in html.lower())

# Parse the HTML table into a DataFrame
df = pd.read_html(
    StringIO(html),
    flavor="lxml",
    header=0,
    index_col=0,
    na_values=["-"]
)[0]

print("DataFrame shape:", df.shape)
df.head()


Requesting NST team table...
Status code: 200
HTML length: 64642
Contains table: True
DataFrame shape: (32, 71)


,Team,GP,TOI/GP,W,L,OTL,ROW,Points,Point %,CF/60,...,LDSA/60,LDSF%,LDGF/60,LDGA/60,LDGF%,LDSH%,LDSV%,SH%,SV%,PDO
1,Chicago Blackhawks,82,48:51,29,39,14,26,72,0.439,50.10,...,11.86,46.49,0.24,0.40,37.21,2.33,96.59,9.56,90.00,0.996
2,Florida Panthers,82,47:16,40,38,4,35,84,0.512,58.11,...,11.05,51.26,0.34,0.45,43.14,2.93,95.94,9.62,89.30,0.989
3,Pittsburgh Penguins,82,48:54,41,25,16,38,98,0.598,57.98,...,10.79,48.90,0.49,0.46,51.56,4.78,95.70,11.13,90.10,1.012
4,New York Rangers,82,49:12,34,39,9,31,77,0.470,54.70,...,12.13,45.74,0.39,0.42,48.15,3.78,96.57,9.56,91.33,1.009
5,Colorado Avalanche,82,48:01,55,16,11,51,121,0.738,70.26,...,11.33,55.98,0.72,0.41,63.51,4.97,96.37,9.84,92.20,1.020


In [ ]:
# Extract the statistics we want from NST
columns = [
    "Team",
    "CF/60",
    "CA/60",
    "xGF/60",
    "xGA/60",
    "CF%",
    "GF%",
    "xGF%",
    "HDCF%",
    "SH%"
]

final_df = df[columns].copy()

# Create rankings for each percentage statistic
rank_columns = ["CF%", "GF%", "xGF%", "HDCF%", "SH%"]

for col in rank_columns:

    # Preserve actual percentage
    final_df[f"{col}_1"] = final_df[col]

    # Replace original column with ranking (1 = best)
    final_df[col] = (
        final_df[col]
        .rank(method="min", ascending=False)
        .astype("Int64")
    )
    
final_df

,Team,CF/60,CA/60,xGF/60,xGA/60,CF%,GF%,xGF%,HDCF%,SH%,CF%_1,GF%_1,xGF%_1,HDCF%_1,SH%_1
1,Chicago Blackhawks,50.10,59.76,2.21,3.02,30,30,32,32,16,45.60,43.27,42.25,40.35,9.56
2,Florida Panthers,58.11,54.24,2.74,2.71,9,24,16,18,15,51.72,47.23,50.21,50.33,9.62
3,Pittsburgh Penguins,57.98,57.58,2.90,2.73,13,7,9,10,2,50.17,53.74,51.52,52.16,11.13
4,New York Rangers,54.70,57.28,2.46,2.55,20,17,22,12,16,48.85,49.51,49.11,51.79,9.56
5,Colorado Avalanche,70.26,53.67,3.21,2.48,2,1,1,3,13,56.69,62.79,56.39,55.37,9.84
6,Los Angeles Kings,61.29,56.09,2.65,2.43,8,12,7,6,27,52.22,52.05,52.19,53.16,8.41
7,Montreal Canadiens,54.74,57.79,2.69,2.85,23,8,23,27,1,48.64,53.49,48.59,46.43,11.30
8,Toronto Maple Leafs,51.67,63.67,2.40,2.86,32,28,29,28,8,44.80,45.36,45.67,46.01,10.28
9,Boston Bruins,57.32,59.84,2.56,2.93,19,5,28,26,6,48.92,54.98,46.66,46.50,10.62
10,Washington Capitals,57.53,58.79,2.88,2.83,15,6,15,16,9,49.46,54.91,50.44,50.59,10.26


In [32]:
def normalize_team(name):
    return (
        name.lower()
        .replace(".", "")
        .replace("é", "e")
        .replace(" ", "")
    )

In [33]:
res = pd.DataFrame()

for away, home, away_goalie, home_goalie in matchups:

    away_df = final_df[
        final_df["Team"].str.contains(
            away,
            case=False,
            na=False
        )
    ]

    home_df = final_df[
        final_df["Team"].str.contains(
            home,
            case=False,
            na=False
        )
    ]

    away_rank = goalie_ranks.get(away_goalie)
    home_rank = goalie_ranks.get(home_goalie)

    away_df = away_df.assign(
        Goalie=away_goalie,
        Goalie_Rank=away_rank
    )

    home_df = home_df.assign(
        Goalie=home_goalie,
        Goalie_Rank=home_rank
    )

    matchup_df = pd.concat(
        [away_df, home_df],
        ignore_index=True
    )

    res = pd.concat(
        [res, matchup_df],
        ignore_index=True
    )

In [34]:
output_file = "../public/result.csv"

res.to_csv(output_file, index=False)

print(f"Saved {len(res)} rows to {output_file}")

Saved 12 rows to ../public/result.csv
